In [36]:
import numpy as np
import pandas as pd

In [37]:
file_path = 'affiliate_orders_masked.csv'
df_raw = pd.read_csv(file_path)

print('Dataset Loaded Successfully!')
print(f'Total Raw Rows   : {df_raw.shape[0]}')
print(f'Total Raw Columns: {df_raw.shape[1]}')

Dataset Loaded Successfully!
Total Raw Rows   : 6889
Total Raw Columns: 47


In [38]:
df_clean = df_raw.copy()

In [39]:
df_clean['order_id'] = pd.factorize(df_clean['ID pesanan'])[0] + 1
df_clean['sku_id'] = pd.factorize(df_clean['ID SKU'])[0] + 1
df_clean['product_id'] = pd.factorize(df_clean['ID produk'])[0] + 1
df_clean['content_id'] = pd.factorize(df_clean['ID Konten'])[0] + 1
df_clean['store_code'] = pd.factorize(df_clean['Kode toko'])[0] + 1

In [40]:
df_clean['order_date'] = pd.to_datetime(
    df_clean['Tanggal pesanan'], format='%d/%m/%Y %H:%M:%S', errors='coerce'
)
df_clean['payout_date'] = pd.to_datetime(
    df_clean['Tanggal penyelesaian pembayaran komisi'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce',
)

In [41]:
df_clean['commission_rate'] = (
    df_clean['Standar']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype(float)
    / 100.0
)
df_clean['commission_rate'] = df_clean['commission_rate'].fillna(0.0)

In [42]:
df_clean['Agensi'] = df_clean['Agensi'].fillna('No Agency')
df_clean['Jenis Konten'] = df_clean['Jenis Konten'].fillna('Unknown')
df_clean['commission_amount'] = df_clean['Total penghasilan akhir'].fillna(0.0)

In [43]:
dim_products = (
    df_clean[['sku_id', 'product_id', 'Nama produk', 'store_code']]
    .drop_duplicates(subset=['sku_id'])
    .copy()
)
dim_products.columns = ['sku_id', 'product_id', 'product_name', 'shop_id']

In [44]:
dim_content = (
    df_clean[['content_id', 'Agensi']].drop_duplicates(subset=['content_id']).copy()
)
dim_content.columns = ['content_id', 'agency_id']

In [45]:
fact_orders = df_clean[[
    'order_id',
    'order_date',
    'sku_id',
    'content_id',
    'Harga',
    'Produk terjual',
    'GMV',
    'Jenis Konten',
    'commission_rate',
    'commission_amount',
    'Status penyelesaian pembayaran pesanan',
    'payout_date',
]].copy()

fact_orders.columns = [
    'order_id',
    'order_date',
    'sku_id',
    'content_id',
    'price',
    'quantity_sold',
    'gmv',
    'content_type',
    'commission_rate',
    'commission_amount',
    'payment_status',
    'payout_date',
]

print('--- Star Schema Table Summary ---')
print(f'dim_products  : {dim_products.shape[0]} unique SKUs')
print(f'dim_content   : {dim_content.shape[0]} unique Content IDs')
print(f'fact_orders   : {fact_orders.shape[0]} transaction rows')

--- Star Schema Table Summary ---
dim_products  : 531 unique SKUs
dim_content   : 284 unique Content IDs
fact_orders   : 6889 transaction rows


In [46]:
check1_sku = dim_products['sku_id'].is_unique
check1_content = dim_content['content_id'].is_unique
print(
    f'Check 1 (Dimension Keys Unique): SKU={check1_sku}, Content={check1_content}'
)

Check 1 (Dimension Keys Unique): SKU=True, Content=True


In [47]:
check2_rows = len(fact_orders) == len(df_raw)
print(
    f'Check 2 (Row Count Match)      : Raw ({len(df_raw)}) vs Fact ({len(fact_orders)}) -> {check2_rows}'
)

Check 2 (Row Count Match)      : Raw (6889) vs Fact (6889) -> True


In [48]:
gmv_diff = abs(fact_orders['gmv'].sum() - df_raw['GMV'].sum())
comm_diff = abs(
    fact_orders['commission_amount'].sum()
    - df_raw['Total penghasilan akhir'].sum()
)
check3_totals = (gmv_diff < 1) and (comm_diff < 1)
print(
    f'Check 3 (Financial Totals Match): GMV Diff={gmv_diff:.2f}, Comm Diff={comm_diff:.2f} -> {check3_totals}'
)

if check1_sku and check1_content and check2_rows and check3_totals:
  print('\nVERDICT: SUCCESS - All 3 Data Integrity Checks Passed!')
else:
  print('\nVERDICT: WARNING - Pipeline Validation Failed!')

Check 3 (Financial Totals Match): GMV Diff=0.00, Comm Diff=0.00 -> True

VERDICT: SUCCESS - All 3 Data Integrity Checks Passed!


In [49]:
dim_products.to_csv('dim_products.csv', index=False)
dim_content.to_csv('dim_content.csv', index=False)
fact_orders.to_csv('fact_orders.csv', index=False)

print(
    'All 3 relational CSV files successfully exported with clean ISO dates & uncollided IDs!'
)

All 3 relational CSV files successfully exported with clean ISO dates & uncollided IDs!
